In [ ]:
from pathlib import Path
import pandas as pd
import json
import joblib

from sklearn.preprocessing import RobustScaler

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_SPLIT = "CIC18__split__v2"
NOMBRE_SCALED = "CIC18__scaled__v2"

RUTA_SPLIT = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SCALED

# ===== ARCHIVOS =====
TRAIN_FILE = f"{NOMBRE_SPLIT}__train.csv"
TEST_FILE = f"{NOMBRE_SPLIT}__test.csv"

TRAIN_OUT = f"{NOMBRE_SCALED}__train.csv"
TEST_OUT = f"{NOMBRE_SCALED}__test.csv"

SCALER_FILE = "scaler.pkl"
REPORT_FILE = f"{NOMBRE_SCALED}_report.json"

# ===== CONFIG =====
LABEL_COL = "LABEL"
SCALER = RobustScaler()

In [ ]:
print("Ruta split:", RUTA_SPLIT)
print("Ruta salida:", RUTA_SALIDA)

In [ ]:
train_path = RUTA_SPLIT / TRAIN_FILE
test_path = RUTA_SPLIT / TEST_FILE

if not train_path.exists():
    raise FileNotFoundError(train_path)

if not test_path.exists():
    raise FileNotFoundError(test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
X_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

X_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

print("Features:", X_train.shape[1])

In [ ]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Columnas numéricas:", len(numeric_cols))

In [ ]:
SCALER.fit(X_train[numeric_cols])

In [ ]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = SCALER.transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = SCALER.transform(X_test[numeric_cols])

In [ ]:
train_scaled = X_train_scaled.copy()
train_scaled[LABEL_COL] = y_train

test_scaled = X_test_scaled.copy()
test_scaled[LABEL_COL] = y_test

In [ ]:
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

train_scaled.to_csv(RUTA_SALIDA / TRAIN_OUT, index=False)
test_scaled.to_csv(RUTA_SALIDA / TEST_OUT, index=False)

print("Train guardado:", TRAIN_OUT)
print("Test guardado:", TEST_OUT)

In [ ]:
joblib.dump(SCALER, RUTA_SALIDA / SCALER_FILE)

print("Scaler guardado en:", SCALER_FILE)

In [ ]:
reporte = {
    "dataset_entrada": NOMBRE_SPLIT,
    "scaler": "RobustScaler",
    "num_features": len(numeric_cols),
    "features_scaled": numeric_cols,
    "train_shape": train_scaled.shape,
    "test_shape": test_scaled.shape
}

with open(RUTA_SALIDA / REPORT_FILE, "w") as f:
    json.dump(reporte, f, indent=2)

print("Reporte guardado")

In [ ]:
print("========== RESUMEN ==========")
print("Scaler:", type(SCALER).__name__)
print("Features escaladas:", len(numeric_cols))
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)
print("=============================")